In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL base del directorio
base_url = "https://www.lucaperrodeasistencia.com/entidades-perro-de-asistencia/?pg={}"

organizaciones = []

# Recorremos las 4 páginas
for pagina in range(1, 5):
    url = base_url.format(pagina)
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Buscamos cada entidad
    entidades = soup.find_all('h3')
    
    for entidad in entidades:
        link = entidad.find('a')
        if link:
            nombre = link.text.strip()
            url_ficha = link.get('href', '')
            # Buscar ubicación (está en el párrafo siguiente)
            ubicacion = ''
            siguiente = entidad.find_next_sibling()
            if siguiente:
                ubicacion = siguiente.text.strip()
            
            organizaciones.append({
                'nombre': nombre,
                'ubicacion': ubicacion,
                'url_ficha': url_ficha
            })
    
    print(f"Página {pagina}: {len(entidades)} entidades encontradas")

# Crear DataFrame
df_orgs = pd.DataFrame(organizaciones)
print(f"\nTotal: {len(df_orgs)} organizaciones")
df_orgs.head(10)

Página 1: 12 entidades encontradas
Página 2: 12 entidades encontradas
Página 3: 12 entidades encontradas
Página 4: 2 entidades encontradas

Total: 38 organizaciones


,nombre,ubicacion,url_ficha
0,ACHUCHO,"Avilés, Asturias",https://www.lucaperrodeasistencia.com/entidade...
1,ARGUS DOGS,Cataluña,https://www.lucaperrodeasistencia.com/entidade...
2,ÁSKAL,"Barcelona, Cataluña",https://www.lucaperrodeasistencia.com/entidade...
3,ASOCIACIÓN CANUCK,Cataluña,https://www.lucaperrodeasistencia.com/entidade...
4,ASOCIACIÓN CRIT – Grupo CTAC,Cataluña,https://www.lucaperrodeasistencia.com/entidade...
5,ASSISTANCE DOGS SPAIN (INTERNATIONAL DETECTOR ...,Cataluña,https://www.lucaperrodeasistencia.com/entidade...
6,BIAK BAT (Consultar con ellos),"Alsasua, Navarra",https://www.lucaperrodeasistencia.com/entidade...
7,CANEM,"Zaragoza, Aragón",https://www.lucaperrodeasistencia.com/entidade...
8,CANNAD,"Villalbilla, Madrid",https://www.lucaperrodeasistencia.com/entidade...
9,CON.TACTO (Sede de DISCAN Galicia),"Vigo, Galicia",https://www.lucaperrodeasistencia.com/entidade...


In [4]:
import time

for i, row in df_orgs.iterrows():
    try:
        response = requests.get(row['url_ficha'],
                               headers={'User-Agent': 'Mozilla/5.0'})
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Buscar tipos — son tarjetas con texto específico
        tipos = []
        tarjetas = soup.find_all('p')
        for tarjeta in tarjetas:
            texto = tarjeta.text.strip()
            if 'Perro Guía' in texto:
                tipos.append('perro_guia')
            if 'Perro Señal' in texto:
                tipos.append('perro_señal')
            if 'Perro de Servicio' in texto:
                tipos.append('perro_servicio')
            if 'Alerta Médica' in texto:
                tipos.append('perro_alerta')
            if 'Espectro Autista' in texto or 'TEA' in texto:
                tipos.append('perro_tea')
        
        df_orgs.at[i, 'tipo_asistencia'] = ';'.join(set(tipos))
        
        # Comunidad autónoma
        ccaa = soup.find(string='COMUNIDAD AUTÓNOMA:')
        if ccaa:
            siguiente = ccaa.find_next('strong')
            if siguiente:
                df_orgs.at[i, 'ccaa'] = siguiente.text.strip()
        
        # Ámbito
        ambito = soup.find(string='ÁMBITO:')
        if ambito:
            siguiente = ambito.find_next('strong')
            if siguiente:
                df_orgs.at[i, 'ambito'] = siguiente.text.strip()
        
        # Web
        web = soup.find(string='Web')
        if web:
            siguiente = web.find_next('p')
            if siguiente:
                df_orgs.at[i, 'web'] = siguiente.text.strip()
        
        time.sleep(0.5)
        
    except Exception as e:
        print(f"Error en {row['nombre']}: {e}")

print("Completado")
df_orgs.head(10)

Completado


,nombre,ubicacion,url_ficha,tipo_asistencia
0,ACHUCHO,"Avilés, Asturias",https://www.lucaperrodeasistencia.com/entidade...,
1,ARGUS DOGS,Cataluña,https://www.lucaperrodeasistencia.com/entidade...,
2,ÁSKAL,"Barcelona, Cataluña",https://www.lucaperrodeasistencia.com/entidade...,
3,ASOCIACIÓN CANUCK,Cataluña,https://www.lucaperrodeasistencia.com/entidade...,
4,ASOCIACIÓN CRIT – Grupo CTAC,Cataluña,https://www.lucaperrodeasistencia.com/entidade...,
5,ASSISTANCE DOGS SPAIN (INTERNATIONAL DETECTOR ...,Cataluña,https://www.lucaperrodeasistencia.com/entidade...,
6,BIAK BAT (Consultar con ellos),"Alsasua, Navarra",https://www.lucaperrodeasistencia.com/entidade...,
7,CANEM,"Zaragoza, Aragón",https://www.lucaperrodeasistencia.com/entidade...,
8,CANNAD,"Villalbilla, Madrid",https://www.lucaperrodeasistencia.com/entidade...,
9,CON.TACTO (Sede de DISCAN Galicia),"Vigo, Galicia",https://www.lucaperrodeasistencia.com/entidade...,


In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time

# Configurar Chrome en modo silencioso (sin abrir ventana visible)
options = webdriver.ChromeOptions()
options.add_argument('--headless')       # no abre ventana del navegador
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# Iniciar el navegador
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

print("Selenium funcionando")

Selenium funcionando


In [6]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

for i, row in df_orgs.iterrows():
    try:
        driver.get(row['url_ficha'])  # abre la ficha en el navegador
        
        # Esperar a que cargue el contenido dinámico
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, 'h1'))
        )
        
        time.sleep(1)  # pausa extra para que cargue el JS
        
        # Buscar tipos de perro
        tipos = []
        elementos = driver.find_elements(By.TAG_NAME, 'p')
        for elem in elementos:
            texto = elem.text.strip()
            if 'Perro Guía' in texto:
                tipos.append('perro_guia')
            if 'Perro Señal' in texto:
                tipos.append('perro_señal')
            if 'Perro de Servicio' in texto:
                tipos.append('perro_servicio')
            if 'Alerta Médica' in texto:
                tipos.append('perro_alerta')
            if 'Espectro Autista' in texto or 'TEA' in texto:
                tipos.append('perro_tea')
        
        # Buscar CCAA
        ccaa_elems = driver.find_elements(By.XPATH, 
            "//p[contains(@class,'label') and contains(text(),'COMUNIDAD')]/following-sibling::p")
        if ccaa_elems:
            df_orgs.at[i, 'ccaa'] = ccaa_elems[0].text.strip()
        
        # Buscar web
        web_elems = driver.find_elements(By.XPATH,
            "//p[contains(text(),'Web')]/following-sibling::p")
        if web_elems:
            df_orgs.at[i, 'web'] = web_elems[0].text.strip()
        
        df_orgs.at[i, 'tipo_asistencia'] = ';'.join(set(tipos))
        print(f"✓ {row['nombre']}")
        
    except Exception as e:
        print(f"✗ Error en {row['nombre']}: {e}")

driver.quit()  # cerrar el navegador al terminar
print("\nCompletado")
df_orgs.head(10)

✓ ACHUCHO
✓ ARGUS DOGS
✓ ÁSKAL
✓ ASOCIACIÓN CANUCK
✓ ASOCIACIÓN CRIT – Grupo CTAC
✓ ASSISTANCE DOGS SPAIN (INTERNATIONAL DETECTOR DOGS TEAM)
✓ BIAK BAT (Consultar con ellos)
✓ CANEM
✓ CANNAD
✓ CON.TACTO (Sede de DISCAN Galicia)
✓ CUENTA CONMIGO PAS
✓ DISCAN
✓ DOGPOINT
✓ EDUCAN
✓ ENTRE HUELLAS (ASKAL Andalucía)
✓ FELIPE VILLANUEVA (Asoc. APORT)
✓ JAS WE CAN (Miembro de ASKAL)
✓ JASKAL
✓ JASKAL
✓ K9 Málaga
✓ KUNÉ
✓ LEALCAN
✓ LIBERCAN (JESUS PASCUA PEREZ)
✓ LOBOS DE NARAIO
✓ MI PERRO SABE
✓ MR. DOGS
✓ Northway K9
✓ OSI Dando Pasos
✓ PERRO ASISTENCIA, José Luis Rubiales
✓ PERROS DE APOYO (AEPA CASTILLA LA MANCHA)
✓ PERROS DE APOYO (AEPA CASTILLA LA MANCHA)
✓ PERROTERAPIA
✓ PERRUNEANDO
✓ RAMALLADAS
✓ RONCESCAN
✓ TERAPICAN (Asociación Canaria de IAP)
✓ TERRACAN
✓ TITA DOG (Mar Pérez Cózar)

Completado


,nombre,ubicacion,url_ficha,tipo_asistencia
0,ACHUCHO,"Avilés, Asturias",https://www.lucaperrodeasistencia.com/entidade...,
1,ARGUS DOGS,Cataluña,https://www.lucaperrodeasistencia.com/entidade...,
2,ÁSKAL,"Barcelona, Cataluña",https://www.lucaperrodeasistencia.com/entidade...,
3,ASOCIACIÓN CANUCK,Cataluña,https://www.lucaperrodeasistencia.com/entidade...,
4,ASOCIACIÓN CRIT – Grupo CTAC,Cataluña,https://www.lucaperrodeasistencia.com/entidade...,
5,ASSISTANCE DOGS SPAIN (INTERNATIONAL DETECTOR ...,Cataluña,https://www.lucaperrodeasistencia.com/entidade...,
6,BIAK BAT (Consultar con ellos),"Alsasua, Navarra",https://www.lucaperrodeasistencia.com/entidade...,
7,CANEM,"Zaragoza, Aragón",https://www.lucaperrodeasistencia.com/entidade...,
8,CANNAD,"Villalbilla, Madrid",https://www.lucaperrodeasistencia.com/entidade...,
9,CON.TACTO (Sede de DISCAN Galicia),"Vigo, Galicia",https://www.lucaperrodeasistencia.com/entidade...,


# Averiguamos como están los datos de los tipos de asistencia de cada asociacion

In [9]:
driver3 = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
driver3.get('https://www.lucaperrodeasistencia.com/entidades-perro-de-asistencia/achucho/')
time.sleep(3)

# Buscar todos los elementos de texto de la página
todos = driver3.find_elements(By.XPATH, '//*[text()]')
for elem in todos:
    texto = elem.text.strip()
    if 'Perro' in texto or 'perro' in texto or 'Guía' in texto or 'Señal' in texto or 'Autista' in texto:
        print(f"Tag: {elem.tag_name} | Clase: {elem.get_attribute('class')} | Texto: {repr(texto)}")

driver3.quit()

Tag: html | Clase: html | Texto: 'Ir al contenido\ninfo@lucaperrodeasistencia.com\nMENÚ\nACHUCHO\nPortada » Entidades perro de asistencia » ACHUCHO\nTipos de perros de asistencia que adiestra esta entidad:\nPerro Guía para Discapacidad Visual\nPerro Señal para Discapacidad Auditiva\nPerro de Servicio para Discapacidad Física\nPerro de Alerta Médica para Desconexión Sensorial\nPerro para Personas con Trastorno del Espectro Autista\nCOMUNIDAD AUTÓNOMA:\nASTURIAS\nSE ENCUENTRAN EN:\nAvilés, Asturias\nÁMBITO:\nLocal/Provincia\nContacto\nWeb\nwww.achucho.org/\n← Volver a entidades\nLuca Perro de Asistencia\ninfo@lucaperrodeasistencia.com\nDisponibilidad\nTeléfono: 613 01 03 91 (Solo Whastapp)\n09:00 - 17:00\nDe Lunes a Viernes\nRedes Sociales\nChat en WhatsApp\nEnlaces de Interés\nDeclaración de Accesibilidad\nMapa del Sitio\nArchivos descargables\nPolítica de privacidad\n© Copyright 2026 - Web diseñada por Dalameda\n🐾'
Tag: body | Clase: wp-singular entidad_asistencia-template-default sing

In [10]:
# Scraping correcto con las clases reales
driver4 = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

for i, row in df_orgs.iterrows():
    try:
        driver4.get(row['url_ficha'])
        time.sleep(2)
        
        # Tipos — clase lpa-tipo-txt
        tipos_elems = driver4.find_elements(By.CLASS_NAME, 'lpa-tipo-txt')
        tipos = []
        for elem in tipos_elems:
            texto = elem.text.strip()
            if 'Guía' in texto:
                tipos.append('perro_guia')
            elif 'Señal' in texto:
                tipos.append('perro_señal')
            elif 'Servicio' in texto:
                tipos.append('perro_servicio')
            elif 'Alerta' in texto:
                tipos.append('perro_alerta')
            elif 'Autista' in texto or 'TEA' in texto:
                tipos.append('perro_tea')
        df_orgs.at[i, 'tipo_asistencia'] = ';'.join(tipos)
        
        # Ficha completa — extraer texto del bloque principal
        ficha = driver4.find_element(By.CLASS_NAME, 'lpa-ficha-card').text
        lineas = ficha.split('\n')
        
        for j, linea in enumerate(lineas):
            if 'COMUNIDAD AUTÓNOMA:' in linea and j+1 < len(lineas):
                df_orgs.at[i, 'ccaa'] = lineas[j+1].strip()
            if 'SE ENCUENTRAN EN:' in linea and j+1 < len(lineas):
                df_orgs.at[i, 'ubicacion'] = lineas[j+1].strip()
            if 'ÁMBITO:' in linea and j+1 < len(lineas):
                df_orgs.at[i, 'ambito'] = lineas[j+1].strip()
            if linea.strip() == 'Web' and j+1 < len(lineas):
                df_orgs.at[i, 'web'] = lineas[j+1].strip()
        
        print(f"✓ {row['nombre']} → {';'.join(tipos)}")
        
    except Exception as e:
        print(f"✗ {row['nombre']}: {e}")

driver4.quit()
print("\nCompletado")
df_orgs[['nombre', 'ccaa', 'tipo_asistencia', 'web']].head(10)

✓ ACHUCHO → perro_guia;perro_señal;perro_servicio;perro_alerta;perro_tea
✓ ARGUS DOGS → perro_guia;perro_señal;perro_servicio;perro_alerta;perro_tea
✓ ÁSKAL → perro_señal
✓ ASOCIACIÓN CANUCK → perro_guia;perro_señal;perro_servicio;perro_alerta;perro_tea
✓ ASOCIACIÓN CRIT – Grupo CTAC → perro_tea
✓ ASSISTANCE DOGS SPAIN (INTERNATIONAL DETECTOR DOGS TEAM) → perro_guia;perro_señal;perro_servicio;perro_alerta;perro_tea
✓ BIAK BAT (Consultar con ellos) → 
✓ CANEM → perro_alerta
✓ CANNAD → perro_guia;perro_señal;perro_servicio;perro_alerta;perro_tea
✓ CON.TACTO (Sede de DISCAN Galicia) → perro_guia;perro_señal;perro_servicio;perro_alerta;perro_tea
✓ CUENTA CONMIGO PAS → perro_servicio;perro_tea
✓ DISCAN → perro_guia;perro_señal;perro_servicio;perro_alerta;perro_tea
✓ DOGPOINT → perro_tea
✓ EDUCAN → perro_servicio
✓ ENTRE HUELLAS (ASKAL Andalucía) → perro_señal
✓ FELIPE VILLANUEVA (Asoc. APORT) → perro_guia;perro_señal;perro_servicio;perro_alerta;perro_tea
✓ JAS WE CAN (Miembro de ASKAL) → pe

,nombre,ccaa,tipo_asistencia,web
0,ACHUCHO,ASTURIAS,perro_guia;perro_señal;perro_servicio;perro_al...,www.achucho.org/
1,ARGUS DOGS,CATALUNYA,perro_guia;perro_señal;perro_servicio;perro_al...,www.argusdogs.com/assistencia-i-deteccio
2,ÁSKAL,CATALUNYA,perro_señal,www.askal.es/
3,ASOCIACIÓN CANUCK,CATALUNYA,perro_guia;perro_señal;perro_servicio;perro_al...,www.associaciocanuck.com/index.php/assistance-...
4,ASOCIACIÓN CRIT – Grupo CTAC,CATALUNYA,perro_tea,ctac.cat/ctac/perros-de-asistencia/
5,ASSISTANCE DOGS SPAIN (INTERNATIONAL DETECTOR ...,CATALUNYA,perro_guia;perro_señal;perro_servicio;perro_al...,www.iddtgroup.com
6,BIAK BAT (Consultar con ellos),NAVARRA,,biakbat.eus/
7,CANEM,ARAGÓN,perro_alerta,canemperrosdealerta.com/
8,CANNAD,MADRID,perro_guia;perro_señal;perro_servicio;perro_al...,canadd.org/
9,CON.TACTO (Sede de DISCAN Galicia),GALICIA,perro_guia;perro_señal;perro_servicio;perro_al...,terapiacontacto.es/terapia-con-animales-en-vigo/


In [11]:
# Guardar el CSV completo con todos los datos
df_orgs.to_csv('../universos/recursos_reales/organizaciones_espana.csv',
               index=False,
               encoding='utf-8-sig')

print(f"Guardado: {len(df_orgs)} organizaciones")
print(df_orgs['ccaa'].value_counts())

Guardado: 38 organizaciones
ccaa
MADRID                12
ANDALUCIA              7
CATALUNYA              6
GALICIA                3
CASTILLA Y LEÓN        2
ASTURIAS               1
NAVARRA                1
ARAGÓN                 1
MURCIA                 1
CASTILLA LA MANCHA     1
I. CANARIAS            1
EXTREMADURA            1
C. VALENCIANA          1
Name: count, dtype: int64
